<a href="https://colab.research.google.com/github/zHenryTM/Gerar-Flashcards-por-Imagens/blob/main/gerar_flashcards.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Execute para instalar as dependências necessárias.

!sudo apt update
!pip install -q -U google-genai

In [ ]:
import os
from google import genai
from google.genai import types
from google.colab import files


# Substituir com o nome do arquivo. A imagem deve ser do tipo .jpeg
image_path = '/content/teste.jpeg'

# Substituir com sua chave API
#     Cole a chave dentro das aspas
#     Consulte o arquivo "tutorial_obter_chave_API.md" para obter sua chave API
chave_API = ''


if not os.path.exists(image_path):
    print(f"\nERRO: O arquivo '{image_path}' não foi encontrado. Por favor, verifique o nome do arquivo ou faça o upload da imagem.")
else:
    try:
        with open(image_path, 'rb') as f:
          image_bytes = f.read()

        client_gemini = genai.Client(api_key=chave_API)

        system_instructions = """Você criará flashcards para ajudar um estudande de engenharia de computação
                                 que estuda na Univasf. Crie cards simples, objetivos e diretos."""

        response_ai = client_gemini.models.generate_content(
            model     = "gemini-3.5-flash",
            config    = types.GenerateContentConfig(system_instruction=system_instructions),
            contents  = [
                types.Part.from_bytes(
                  data      = image_bytes,
                  mime_type = 'image/jpeg',
                ),
                """
                Através dessas anotações de uma aula, crie flashcards, agrupe-os em um baralho no anki e
                me devolve como TSV. Não transcreva o texto. Interprete as informações e elabore os flashcards.
                Devolva apenas o TSV e não inclua o ```tst ```.
                """
            ]
        )

        print(response_ai.text)

        with open('flashcards.tsv', 'w') as f:
          f.write(response_ai.text)

        files.download('flashcards.tsv')

    except Exception as e:
        print(f"Ocorreu um erro: {e}")